# Cape Jones — ToonCrafter cartoon interpolation on Kaggle T4 GPU

Interpolates `expressions-v2-noblack.mp4` (106 frames @ 12fps, 1080x1920,
**no black frames**) to 24fps using **ToonCrafter** (SIGGRAPH Asia 2024,
CUHK & Tencent, Apache-2.0) on a **Tesla T4 GPU**.

**Why ToonCrafter instead of RIFE?** RIFE was trained on real video — its
optical flow can't track motion in flat, textureless cartoon regions (white
suit, purple cape), so it blends instead of warping = ghosting. ToonCrafter
is a diffusion model **trained on cartoons** — it handles the exaggerated
non-linear motion and dis-occlusion in animation natively.

**Before running:** Settings (right panel) -> **Accelerator: GPU T4 x2**,
**Internet: ON**. Then **Run All**.

Pipeline in this notebook:
1. GPU check (must be Tesla T4)
2. Clone ToonCrafter + pip install deps (+ FP16 execution patch)
3. Download FP16 model.ckpt from HuggingFace (~3.2GB, ~12GB VRAM — fits T4)
4. Download input video from GitHub (SHA-256 verified)
5. Extract 106 frames, resize to 320x512 portrait, build 105 keyframe pairs
6. ToonCrafter inference on CUDA (105 pairs x 8 frames, take middle frame idx 4)
7. Interleave originals + middles -> 211 frames @ 24fps MP4 -> `/kaggle/working/`


In [ ]:
# === GPU check ===
import subprocess
r = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                    "--format=csv"], capture_output=True, text=True)
print(r.stdout.strip() or "NO GPU VISIBLE -- enable Accelerator: GPU in Settings!")
assert "Tesla T4" in r.stdout, "Expected a Tesla T4"


In [ ]:
# === Bulletproof saver setup (skip if queue already injected it) ===
# save_outputs.py must sit next to this notebook (queue ships it automatically;
# for manual browser runs, upload it alongside the .ipynb).
import os, sys
if "_saver" not in dir():
    for _p in ("/kaggle/working", os.getcwd()):
        if _p not in sys.path:
            sys.path.insert(0, _p)
    try:
        from save_outputs import RenderSaver
        _saver = RenderSaver(dataset_slug=os.environ.get("KAGGLE_SAVE_DATASET"))
        if _saver.enabled:
            _saver.start_watchdog()
            print(f"bulletproof saver ACTIVE -> {_saver.dataset_slug}")
        else:
            print("bulletproof saver: KAGGLE_SAVE_DATASET not set — "
                  "set it or outputs rely on Save & Run All only")
    except Exception as e:
        _saver = None
        print(f"bulletproof saver UNAVAILABLE: {e}")
else:
    print("saver already active (queue-injected)")


In [ ]:
# === Clone ToonCrafter + install deps ===
import subprocess, sys, os
if not os.path.isdir("/kaggle/working/ToonCrafter"):
    print("Cloning ToonCrafter...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Doubiiu/ToonCrafter.git",
                    "/kaggle/working/ToonCrafter"], check=True)
else:
    print("ToonCrafter already cloned.")
%cd /kaggle/working/ToonCrafter
# Kaggle ships torch+torchvision; install the rest (skip torch to save time)
print("Installing deps...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                # NOTE (2026-10-16): decord REMOVED — no cp313 wheels (abandoned
                # since 2022), and it's only used in training/eval scripts,
                # never in inference. transformers bumped 4.25.1 -> 4.48.3
                # because 4.25.1 pins tokenizers<0.14 which has no cp313 wheels
                # (build from source fails). 4.48.3 needs tokenizers>=0.21
                # which ships cp39-abi3 wheels (work on 3.13). Verified:
                # CLIPTokenizer/CLIPTextModel/T5Tokenizer/T5EncoderModel all
                # import fine on 4.48.3.
                "einops==0.3.0", "omegaconf==2.1.1",
                "pytorch_lightning==1.9.3", "transformers==4.48.3",
                "open_clip_torch==2.22.0", "xformers", "timm",
                "kornia", "av", "moviepy",
                # v3 (2026-10-16): added missing deps from official requirements.txt
                "imageio==2.9.0", "opencv-python", "Pillow==9.5.0",
                "PyYAML==6.0", "tqdm==4.65.0", "safetensors"],
               check=True)
print("Deps installed.")


In [ ]:
# === Patch ToonCrafter for FP16 execution (fits T4 VRAM) ===
# v4 (2026-10-16): The official README's 12GB figure assumes fp16 execution
# (the ComfyUI path runs the model in fp16/bf16). Stock inference.py loads the
# weights into an fp32 model — load_state_dict casts fp16->fp32, so weights
# alone would eat ~6.4GB and total VRAM could push past 16GB on a T4.
# We patch run_inference() to half() the model right after checkpoint load.
# Kijai's own ComfyUI wrapper does exactly this (nodes.py: model.to(precision)).
import os
p = "/kaggle/working/ToonCrafter/scripts/evaluation/inference.py"
src = open(p).read()
old = "    model = load_model_checkpoint(model, args.ckpt_path)\n    model.eval()"
new = ("    model = load_model_checkpoint(model, args.ckpt_path)\n"
       "    model = model.half()  # v4: FP16 execution -> ~12GB VRAM on T4\n"
       "    model.eval()")
assert "model = model.half()" not in src, "FP16 patch already applied"
assert old in src, "PATCH ANCHOR NOT FOUND — upstream inference.py changed, fix me"
open(p, "w").write(src.replace(old, new))
print("FP16 execution patch applied to inference.py")


In [ ]:
# === ToonCrafter model weights: FP16 pruned (~3.2GB, ~12GB VRAM, fits T4) ===
# v4 (2026-10-16): Switched from full model.ckpt (10.5GB, ~24-27GB VRAM — OOM
# on T4) to Kijai's pruned FP16 (3.18GB, ~12GB VRAM with fp16 execution).
# See docs/research/TOONCRAFTER-FULL-STACK.md.
# CORRECT FILENAME (verified via HF API siblings + Kijai's ComfyUI wrapper):
#   tooncrafter_512_interp-pruned-fp16.safetensors
# The official ToonCrafter README links tooncrafter_512_interp-fp16.safetensors
# (underscore) — that link is STALE and 404s. Do NOT "fix" this back to it.
# Converted via production/kaggle/tooncrafter/convert_safetensors_to_ckpt.py
import subprocess, os, shutil
ckpt_dir = "/kaggle/working/ToonCrafter/checkpoints/tooncrafter_512_interp_v1"
os.makedirs(ckpt_dir, exist_ok=True)
ckpt_path = os.path.join(ckpt_dir, "model.ckpt")  # inference.py expects this name
CACHED = "/kaggle/input/tooncrafter-weights-fp16/model.ckpt"
def _ok(p):
    # FP16 pruned model is 3,175,466,358 bytes on HF; torch.save adds a few %
    # pickle overhead. Accept 2.8-3.8GB. (v3's 2.0-3.0GB range was wrong —
    # it would have REJECTED the genuine file.)
    return os.path.exists(p) and 2_800_000_000 < os.path.getsize(p) < 3_800_000_000
if _ok(ckpt_path):
    print(f"FP16 model.ckpt already present ({os.path.getsize(ckpt_path)/1e9:.1f} GB)")
elif os.path.exists(CACHED) and _ok(CACHED):
    print(f"Copying FP16 model.ckpt from dataset cache ({os.path.getsize(CACHED)/1e9:.1f} GB)...")
    shutil.copy(CACHED, ckpt_path)
    print("Cached FP16 weights linked. VRAM required: ~12GB (fits T4)")
else:
    print("Dataset cache miss — downloading FP16 safetensors (3.2GB) from HuggingFace...")
    subprocess.run(["wget", "-q", "--show-progress",
                    "https://huggingface.co/Kijai/DynamiCrafter_pruned/resolve/main/tooncrafter_512_interp-pruned-fp16.safetensors",
                    "-O", "/kaggle/working/tooncrafter_fp16.safetensors"], check=True)
    print("Converting safetensors -> .ckpt...")
    from safetensors.torch import load_file
    import torch
    sd = load_file("/kaggle/working/tooncrafter_fp16.safetensors", device="cpu")
    torch.save({"state_dict": sd}, ckpt_path)
    os.remove("/kaggle/working/tooncrafter_fp16.safetensors")
    assert _ok(ckpt_path), "converted model.ckpt size mismatch"
    print(f"Converted ({os.path.getsize(ckpt_path)/1e9:.1f} GB). VRAM: ~12GB (fits T4)")


In [ ]:
# === Download input video from GitHub (SHA-256 verified) ===
import subprocess, hashlib, os
url = "https://raw.githubusercontent.com/mhvnsnt/cape-jones/kaggle-tooncrafter/production/kaggle/tooncrafter/input-12fps-noblack.mp4"
out = "/kaggle/working/input-12fps-noblack.mp4"
print(f"Downloading {url} ...")
subprocess.run(["wget", "-q", url, "-O", out], check=True)
h = hashlib.sha256(open(out, "rb").read()).hexdigest()
print(f"sha256: {h}")
assert h == "c039dff8332f4e9ae697e81eb36f709dc38b0303eb830a2d3e7903fc91b1e19a", f"SHA MISMATCH! got {h}"
print("Video verified OK.")


In [ ]:
# === Extract frames + build 105 keyframe pairs ===
# ToonCrafter interp mode: each sample = 2 keyframes -> generates 8 frames (tuned).
# We need 105 pairs from 106 frames. Portrait: resize to 320x512.
import subprocess, os, shutil

work = "/kaggle/working/tc_input"
frames_dir = os.path.join(work, "frames")
prompt_dir = os.path.join(work, "prompts", "cape_jones")
for d in (frames_dir, prompt_dir):
    os.makedirs(d, exist_ok=True)

# 1. Extract 106 frames at original res
print("Extracting frames...")
subprocess.run(["ffmpeg", "-y", "-v", "error",
                "-i", "/kaggle/working/input-12fps-noblack.mp4",
                "-vsync", "0",
                os.path.join(frames_dir, "frame_%05d.png")], check=True)
n = len([f for f in os.listdir(frames_dir) if f.endswith(".png")])
print(f"{n} frames extracted (expect 106)")
assert n == 106, f"Expected 106 frames, got {n}"

# 2. Resize to 320x512 portrait + build pairs
#    Pair i = (frame_i, frame_{i+1}) -> named pair_{i:03d}_a.png / _b.png
print("Resizing to 320x512 and building 105 pairs...")
prompt_lines = []
for i in range(105):
    for tag, src_idx in (("a", i), ("b", i + 1)):
        src = os.path.join(frames_dir, f"frame_{src_idx+1:05d}.png")
        dst = os.path.join(prompt_dir, f"pair_{i:03d}_{tag}.png")
        subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", src,
                        "-vf", "scale=320:512", dst], check=True)
    # prompt file: one line per sample (ToonCrafter uses it for n_samples count)
    prompt_lines.append(f"cape jones cartoon character, pair {i}")

with open(os.path.join(prompt_dir, "prompts.txt"), "w") as f:
    f.write("\n".join(prompt_lines) + "\n")
print(f"105 pairs built in {prompt_dir}")


In [ ]:
# === ToonCrafter inference on CUDA ===
# 105 pairs -> 8 frames each (TUNED: was 16). We take the middle frame (idx 4).
# ddim_steps=12 (TUNED: was 25). bs=1 for VRAM safety.
# --n_samples 1: n_samples = samples PER PROMPT in this script (default 1).
# The prompt loop already iterates all 105 prompts; 105 here would generate
# 105 samples x 105 prompts = 11,025 videos. Do NOT set to 105.
# See TUNING.md for speed/quality tradeoff analysis.
import subprocess, time, os

t0 = time.time()
cmd = [
    "python3", "scripts/evaluation/inference.py",
    "--seed", "123",
    "--ckpt_path", "checkpoints/tooncrafter_512_interp_v1/model.ckpt",
    "--config", "configs/inference_512_v1.0.yaml",
    "--savedir", "/kaggle/working/tc_results",
    "--n_samples", "1",
    "--bs", "1",
    "--height", "512",
    "--width", "320",
    "--unconditional_guidance_scale", "7.5",
    "--ddim_steps", "12",
    "--ddim_eta", "1.0",
    "--prompt_dir", "/kaggle/working/tc_input/prompts/cape_jones",
    "--text_input",
    "--video_length", "8",
    "--frame_stride", "10",
    "--timestep_spacing", "uniform_trailing",
    "--guidance_rescale", "0.7",
    "--perframe_ae",
    "--interp",
]
print("Running ToonCrafter inference (105 pairs, this takes a while)...")
print(" ".join(cmd))
r = subprocess.run(cmd, cwd="/kaggle/working/ToonCrafter")
elapsed = (time.time() - t0) / 60
print(f"\nToonCrafter wall time = {elapsed:.1f} min")
assert r.returncode == 0, "ToonCrafter inference FAILED"
print("Inference complete.")


In [ ]:
# === Extract middle frames + interleave + assemble 24fps MP4 ===
# v4 (2026-10-16): inference.py's save_results_seperate() writes ONE 8-frame
# MP4 per prompt into <savedir>/samples_separate/ (the "samples" dir stays
# empty — save_results() is commented out upstream). Files are named
#   samples_separate/pair_{i:03d}_a_sample0.mp4   (8 frames @ fps=10)
# We dump each MP4's 8 frames, take frame idx 4 (the true temporal middle),
# and interleave: [orig_0, mid_0, orig_1, mid_1, ...] = 211 frames @ 24fps.
import os, glob, shutil, subprocess, tempfile
from PIL import Image

sep_dir = "/kaggle/working/tc_results/samples_separate"
mp4s = sorted(glob.glob(os.path.join(sep_dir, "*_sample0.mp4")))
print(f"Found {len(mp4s)} sample MP4s (expect 105)")
assert len(mp4s) == 105, f"Expected 105 samples, got {len(mp4s)}"

out_dir = "/kaggle/working/final_frames"
os.makedirs(out_dir, exist_ok=True)
frames_dir = "/kaggle/working/tc_input/frames"
tmpd = tempfile.mkdtemp(prefix="tc_mid_")

out_idx = 0
for i, mp4 in enumerate(mp4s):
    # original frame i (full-res 1080x1920)
    orig = os.path.join(frames_dir, f"frame_{i+1:05d}.png")
    shutil.copy(orig, os.path.join(out_dir, f"out_{out_idx:05d}.png"))
    out_idx += 1
    # dump the 8 frames of this sample MP4, take idx 4 (0-based), upscale
    subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", mp4,
                    os.path.join(tmpd, f"s{i:03d}_%d.png")], check=True)
    dumped = sorted(glob.glob(os.path.join(tmpd, f"s{i:03d}_*.png")))
    assert len(dumped) == 8, f"Sample {i}: expected 8 frames, got {len(dumped)}"
    mid = Image.open(dumped[4]).convert("RGB").resize((1080, 1920), Image.LANCZOS)
    mid.save(os.path.join(out_dir, f"out_{out_idx:05d}.png"))
    out_idx += 1
    for f in dumped:
        os.remove(f)
shutil.rmtree(tmpd, ignore_errors=True)

# final original frame (frame 106)
shutil.copy(os.path.join(frames_dir, "frame_00106.png"),
            os.path.join(out_dir, f"out_{out_idx:05d}.png"))
out_idx += 1

print(f"{out_idx} frames assembled (expect 211)")
assert out_idx == 211, f"Expected 211 frames, got {out_idx}"

# assemble 24fps MP4
subprocess.run(["ffmpeg", "-y", "-v", "error",
                "-framerate", "24", "-i", os.path.join(out_dir, "out_%05d.png"),
                "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18",
                "/kaggle/working/cape-jones-expressions-TOONCRAFTER-24fps.mp4"],
               check=True)
print("Wrote /kaggle/working/cape-jones-expressions-TOONCRAFTER-24fps.mp4")

# verify
r = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0",
                    "-show_entries", "stream=nb_frames,r_frame_rate,width,height",
                    "-of", "default=noprint_wrappers=1",
                    "/kaggle/working/cape-jones-expressions-TOONCRAFTER-24fps.mp4"],
                   capture_output=True, text=True)
print(r.stdout)


In [ ]:
# === Bulletproof final checkpoint ===
# Publishes the finished MP4 as a Kaggle dataset version NOW — never rely on
# Quick Save (it preserves zero output files). Skip if queue injected its own.
_saver = globals().get("_saver")
_out = "/kaggle/working/cape-jones-expressions-TOONCRAFTER-24fps.mp4"
if _saver and getattr(_saver, "enabled", False):
    import os
    if os.path.exists(_out):
        _saver.checkpoint("tooncrafter-final", [_out])
        print(f"checkpointed {_out}")
    else:
        print(f"WARNING: expected output missing: {_out}")
    _saver.stop_watchdog()
elif "_saver" in dir() and _saver is None:
    print("saver unavailable — output relies on Save & Run All")
else:
    print("queue-injected final checkpoint will handle saving")
